In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import io
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

session = get_active_session()

# ---------------------------------------------------
# 1️⃣ Load ML Feature Table
# ---------------------------------------------------

df = session.table("SALES_DB.PROCESSED_SCHEMA.CUSTOMER_ML_FEATURES").to_pandas()

df = df.dropna()

# ---------------------------------------------------
# 2️⃣ Feature / Target Split
# ---------------------------------------------------

X = df[["AGE", "GENDER", "REGION"]]
y = df["STATUS"]

print("Class distribution:")
print(y.value_counts())

# ---------------------------------------------------
# 3️⃣ One-Hot Encode
# ---------------------------------------------------

X_encoded = pd.get_dummies(X)

feature_columns = X_encoded.columns.tolist()

# ---------------------------------------------------
# 4️⃣ Train/Test Split
# ---------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# ---------------------------------------------------
# 5️⃣ Train Model
# ---------------------------------------------------

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# ---------------------------------------------------
# 6️⃣ Evaluate
# ---------------------------------------------------

preds = model.predict(X_test)
accuracy = accuracy_score(y_test, preds)

print("Model Accuracy:", accuracy)

# ---------------------------------------------------
# 7️⃣ Save Model + Feature Schema
# ---------------------------------------------------

model_bundle = {
    "model": model,
    "features": feature_columns
}

model_bytes = io.BytesIO()
joblib.dump(model_bundle, model_bytes)
model_bytes.seek(0)

session.file.put_stream(
    model_bytes,
    "@SALES_DB.PROCESSED_SCHEMA.ML_STAGE/customer_model.joblib",
    overwrite=True
)

print("Model + feature metadata saved successfully.")

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE FUNCTION SALES_DB.PROCESSED_SCHEMA.PREDICT_CUSTOMER_STATUS(
    AGE NUMBER,
    GENDER STRING,
    REGION STRING
)
RETURNS FLOAT
LANGUAGE PYTHON
RUNTIME_VERSION = 3.10
HANDLER = 'predict'
PACKAGES = ('scikit-learn', 'pandas', 'joblib')
IMPORTS = ('@SALES_DB.PROCESSED_SCHEMA.ML_STAGE/customer_model.joblib')
AS
$$
import joblib
import pandas as pd
import sys
import os

import_dir = sys._xoptions["snowflake_import_directory"]
model_path = os.path.join(import_dir, "customer_model.joblib")

bundle = joblib.load(model_path)
model = bundle["model"]
feature_columns = bundle["features"]

def predict(age, gender, region):

    df = pd.DataFrame([[age, gender, region]],
                      columns=["AGE", "GENDER", "REGION"])

    df = pd.get_dummies(df)

    # Add missing columns
    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0

    df = df[feature_columns]

    return float(model.predict_proba(df)[0][1])
$$;